# exp-03 — does running the checks through one entry skill degrade them?

Reads the runs under `experiments/runs/exp-03-checklist-entry/`, scores every
leaf that is not scored yet, and compares each check run under the entry skill
`do-fig-checklist` (**D**) against the same check dispatched on its own.

**This notebook spends nothing.** The sessions are produced by

```bash
python experiments/exp-03-checklist-entry/run.py
```

which is 1,520 sessions: 380 fan-out, 1,140 per-check controls.

Scoring *is* done here, for exp-02's reason: a rollup cached at run time is one
that nothing invalidates when a threshold moves.

The claims this is evidence for live in
[`thinking/experiments/exp-03-checklist-entry.md`](../../thinking/experiments/exp-03-checklist-entry.md).
Nothing here decides whether the experiment succeeded.

## The four conditions

Two arrangements of each check, each run two ways.

| condition | checklist | arm directory | sessions per figure |
|---|---|---|---|
| `A\|B\|C_i` | `fig-checklist-exp03-per-check` | `per-check/<check>/pinned` | 3, one per check |
| `A\|B <- C_i` | `fig-checklist-exp03-per-check` | `per-check/<check>/<check>@v3` | 3 |
| `A\|B\|C_i <- D` | `fig-checklist-exp03` | `fan-out/pinned` | 1, all three checks |
| `A\|B <- C_i <- D` | `fig-checklist-exp03` | `fan-out/<all three>@v3` | 1 |

Every skill is byte-identical between the two checklists, and neither carries
`identify-panels`. The fan-out is scored by D's contract, which is derived from
the three per-check contracts and verified to score each check identically
(`experiments/exp-03-checklist-entry/build_contract.py`). Its frames are
rewritten into per-check terms below, so one set of code reads both.

## The gold this experiment was scored against

exp-03 was run and scored on 2026-09-28/29, against the gold as of 2026-09-28. The gold has been curated and migrated since — the contract cleanup (`thinking/plans/2026-09-30-contract-cleanup.md`) rewrote tokens and decisions — so re-scoring at head would change these numbers without saying so. This cell pins the scorer to **`gold-v1`**, a git tag, read from a gold-only snapshot (`soda_mmqc/gold.py`). It must run before anything else imports `soda_mmqc`; to re-score under the current gold, which is exploratory, remove it and restart the kernel.

In [ ]:
from soda_mmqc.gold import pin_gold

GOLD_TAG = "gold-v1"
pin_gold(GOLD_TAG)

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from soda_mmqc.core.run_layout import BASELINE_ARM, iter_leaves
from soda_mmqc.core.scoring import ANALYSIS_FILENAME, score_check
from soda_mmqc.reporting.styles import (
    LAYER1_ORDER,
    LAYER2_BINARY_ORDER,
    LAYER2_GRADED_ORDER,
    LAYER_S_ORDER,
)
from soda_mmqc.reporting import (
    arm_contrast,
    layer1_counts_by_example,
    layer2_counts,
    layer_s_counts_by_example,
    load_run_root,
    non_response_counts,
    plot_stacked_counts,
    rate_contrast,
    scores_frame,
)

FAN_OUT = "fig-checklist-exp03"
PER_CHECK = "fig-checklist-exp03-per-check"
ENTRY = "do-fig-checklist"
CLASSIFY = "classify-panels"
CHECKS = ("micrograph-scale-bar", "individual-data-points", "error-bars-defined")

#: Point this at `RUNS / "smoke"` to exercise the notebook on the smoke test.
RUNS = Path("../../experiments/runs/exp-03-checklist-entry")
ROOT = RUNS
from soda_mmqc.config import gold_root

#: Where gold is read: the pinned snapshot above, not the live tree.
GOLD = gold_root()

#: Margins, fixed in the note before the run.
DELTA = {"layer S": 0.0125, "layer 1": 0.02, "layer 2": 0.02}

#: Seeded so the intervals are reproducible.
SEED = 20260927

MONOLITH, DELEGATING = "A|B|C_i", "A|B <- C_i"
#: Each fan-out condition, and the per-check control it is compared with.
PAIRS = {f"{MONOLITH} <- D": MONOLITH, f"{DELEGATING} <- D": DELEGATING}
CONDITION_ORDER = (MONOLITH, f"{MONOLITH} <- D", DELEGATING, f"{DELEGATING} <- D")


def condition(part: str, arm: str) -> str:
    """Name a run by its topology rather than by what moved off the pin."""
    arrangement = MONOLITH if arm == BASELINE_ARM else DELEGATING
    return f"{arrangement} <- D" if part == "fan-out" else arrangement


def owning_check(name: str) -> str | None:
    """The check a fan-out list key or property belongs to."""
    for check in CHECKS:
        if name == check or name.startswith((f"{check}[", f"{check}.")):
            return check
    return None


def to_per_check(frame: pd.DataFrame, part: str) -> pd.DataFrame:
    """Rewrite a frame into per-check terms, and label its condition.

    A fan-out frame is one check, `do-fig-checklist`, whose properties are
    `<check>[].x` and whose list keys are `<check>`. Rewritten, it reads
    exactly like a per-check frame -- `check`, `outputs[].x`, `outputs` -- so
    the two can be concatenated and paired.
    """
    frame = frame.copy()
    if part == "fan-out":
        key = "property" if "property" in frame else "list_key"
        frame["check"] = [owning_check(v) for v in frame[key]]
        if frame["check"].isna().any():
            raise ValueError(f"fan-out {key}s owned by no check: "
                             f"{sorted(frame.loc[frame['check'].isna(), key].unique())}")
        for column in ("property", "list_key"):
            if column in frame:
                frame[column] = [
                    "outputs" + value[len(check):]
                    for value, check in zip(frame[column], frame["check"])
                ]
    frame["condition"] = [condition(part, arm) for arm in frame["arm"]]
    return frame


def run_roots():
    """(part, checklist, check, root) for every run root under ROOT."""
    yield "fan-out", FAN_OUT, ENTRY, ROOT / "fan-out"
    for check in CHECKS:
        yield "per-check", PER_CHECK, check, ROOT / "per-check" / check


print(ROOT.resolve())

## Score every leaf that is not scored yet

As in exp-02: idempotent, and it refuses to trust an `analysis.json` older than
the predictions under it — **or older than the gold it was scored against**. Gold
is curated after runs as well as before them, and a fix to one label must reach
every score that used it; exp-02's notebook checks predictions only.

In [ ]:
stale = []
for part, checklist, check, root in run_roots():
    if not root.is_dir():
        print(f"no runs yet under {root}")
        continue
    for arm, replicate, leaf in iter_leaves(root):
        analysis = leaf / ANALYSIS_FILENAME
        if not analysis.is_file():
            score_check(checklist, check, leaf, save=True)
            print(f"scored {part} / {check} / {arm} / rep-{replicate:02d}")
            continue
        predictions = list(leaf.rglob("prediction.json"))
        gold = [
            GOLD / p.parent.relative_to(leaf) / "checks" / check / "expected_output.json"
            for p in predictions
        ]
        newest = max(
            (p.stat().st_mtime for p in predictions + gold if p.is_file()),
            default=0.0,
        )
        if newest > analysis.stat().st_mtime:
            stale.append(leaf)

if stale:
    raise RuntimeError(
        "predictions or gold are newer than their analysis in "
        f"{len(stale)} leaf/leaves; delete those analysis.json and re-run "
        f"this cell:\n  " + "\n  ".join(str(p) for p in stale)
    )
print("all leaves scored, none stale")

## Endpoint 0 — did D dispatch?

Read before any score. `skill_trace.json` lists every skill a session invoked,
in order. For a fan-out session:

- **dispatch** — all three check skills invoked, and per check whether it was;
- **reach** — in `A|B <- C_i <- D`, whether `classify-panels` was invoked;
- **reuse** — how many times `classify-panels` was invoked in the session: once
  means one classification served all three checks;
- **order** — the order in which the checks were first invoked.

A `classify-panels` call is attributed to the check invoked most recently before
it. That is an inference from order, not something the trace records.

**Reported, not a gate**, as in exp-02: a non-inferior check with partial
dispatch is *safe but optional*, and a degraded one is read with its rate.

In [ ]:
def read_trace(path: Path) -> list[str]:
    try:
        entries = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return []
    return [e.get("skill") for e in entries if isinstance(e, dict) and e.get("skill")]


rows = []
for part, checklist, check, root in run_roots():
    if not root.is_dir():
        continue
    for arm, replicate, leaf in iter_leaves(root):
        for trace in leaf.rglob("intermediates/skill_trace.json"):
            skills = read_trace(trace)
            called = [s for s in skills if s in CHECKS]
            reached = {c: 0 for c in CHECKS}
            current = None
            for s in skills:
                if s in CHECKS:
                    current = s
                elif s == CLASSIFY and current:
                    reached[current] += 1
            owners = CHECKS if part == "fan-out" else (check,)
            expected = {*owners, ENTRY} if part == "fan-out" else {check}
            if arm != BASELINE_ARM:
                expected.add(CLASSIFY)
            for owner in owners:
                rows.append({
                    "part": part,
                    "condition": condition(part, arm),
                    "check": owner,
                    "replicate": replicate,
                    "example": str(trace.parent.parent.relative_to(leaf)),
                    "dispatched": owner in called,
                    "all_dispatched": set(owners) <= set(called),
                    "reached": reached[owner] > 0,
                    "classify_calls": skills.count(CLASSIFY),
                    "order": " > ".join(dict.fromkeys(called)),
                    "position": list(dict.fromkeys(called)).index(owner) + 1
                    if owner in called else None,
                    "spontaneous": tuple(sorted(set(skills) - expected)),
                })

traces = pd.DataFrame(rows)
(
    traces.groupby(["condition", "check"])[["dispatched", "reached"]]
    .mean()
    .join(traces.groupby(["condition", "check"]).size().rename("sessions"))
    .reindex(CONDITION_ORDER, level=0)
)

In [ ]:
# One row per fan-out session: did it reach every check, and how often did it
# classify? `classify_calls` = 1 in the delegating arm is one classification
# shared by three checks; 3 is one per check.
fan_sessions = traces[traces["part"] == "fan-out"].drop_duplicates(
    ["condition", "replicate", "example"]
)
print("all three checks dispatched:")
print(fan_sessions.groupby("condition")["all_dispatched"].agg(["mean", "sum", "count"]))
print()
print("classify-panels calls per fan-out session:")
print(fan_sessions.groupby(["condition", "classify_calls"]).size().unstack(fill_value=0))

In [ ]:
# The order D ran the checks in. If one order dominates, a check's position and
# its identity are confounded -- see the note's threats to validity.
fan_sessions.groupby(["condition", "order"]).size().rename("sessions").reset_index()

In [ ]:
# Skills invoked that the condition never named. An empty table means every
# invocation was one the arrangement asked for.
exploded = traces.explode("spontaneous").dropna(subset=["spontaneous"])
(
    exploded.drop_duplicates(["part", "condition", "replicate", "example", "spontaneous"])
    .groupby(["condition", "spontaneous"]).size().rename("sessions").reset_index()
)

## One row per property, per replicate, per example

Both run types are loaded, the fan-out rewritten into per-check terms, and
concatenated. From here on a row is a `(condition, check, …)` and nothing
downstream knows which checklist produced it.

In [ ]:
frames, layer_s, layer_1, layer_2, non_response = [], [], [], [], []
for part, checklist, check, root in run_roots():
    if not root.is_dir():
        continue
    runs = load_run_root(root, checklist=checklist, check=check)
    frames.append(to_per_check(scores_frame(runs), part))
    layer_s.append(to_per_check(layer_s_counts_by_example(runs), part))
    layer_1.append(to_per_check(layer1_counts_by_example(runs), part))
    layer_2.append(to_per_check(layer2_counts(runs), part))
    non_response.append(to_per_check(non_response_counts(runs), part))

scores = pd.concat(frames, ignore_index=True)
layer_s = pd.concat(layer_s, ignore_index=True)
layer_1 = pd.concat(layer_1, ignore_index=True)
layer_2 = pd.concat(layer_2, ignore_index=True)
non_response = pd.concat(non_response, ignore_index=True)

print(f"{len(scores)} score rows, {scores['check'].nunique()} checks, "
      f"{scores['replicate'].nunique()} replicates, "
      f"{scores['condition'].nunique()} conditions")
scores.head()

## Non-response and failures, before any mean

`outputs: []` scores as a fully missing row set, as in exp-02. A session that
*failed* wrote no prediction at all — and a failed fan-out session loses all
three checks at once, so failures are counted per session as well as per check.

In [ ]:
(
    non_response.groupby(["check", "condition"])["empty"]
    .sum()
    .unstack(fill_value=0)
    .reindex(columns=CONDITION_ORDER)
)

In [ ]:
def benchmark(checklist: str, check: str) -> list[str]:
    path = Path("../../soda_mmqc/data/checklist") / checklist / check / "benchmark.json"
    return json.loads(path.read_text(encoding="utf-8"))["examples"]


missing = []
for part, checklist, check, root in run_roots():
    if not root.is_dir():
        continue
    wanted = benchmark(checklist, check)
    for arm, replicate, leaf in iter_leaves(root):
        absent = sum(not (leaf / ex / "prediction.json").is_file() for ex in wanted)
        missing.append({
            "part": part, "condition": condition(part, arm), "check": check,
            "replicate": replicate, "sessions_without_prediction": absent,
            "sessions": len(wanted),
        })
missing = pd.DataFrame(missing)
missing.groupby(["part", "condition", "check"])[
    ["sessions_without_prediction", "sessions"]
].sum()

## The outcomes, condition beside condition

Counts, not means, summed over examples and replicates — exp-02's figures, with
four conditions per check. Read the slivers against the gate tables, not against
bar heights.

In [ ]:
plot_stacked_counts(
    layer_s,
    group="check",
    category="outcome",
    series="condition",
    order=LAYER_S_ORDER,
    series_order=CONDITION_ORDER,
    ylabel="rows",
    title="Layer S row outcomes, summed over examples and replicates",
)

In [ ]:
plot_stacked_counts(
    layer_1,
    group="check",
    category="layer1",
    series="condition",
    order=LAYER1_ORDER,
    series_order=CONDITION_ORDER,
    ylabel="instances",
    title="Layer 1 applicability calls, summed over properties, examples and replicates",
)

In [ ]:
plot_stacked_counts(
    layer_2[layer_2["layer2"].isin(LAYER2_BINARY_ORDER)],
    group="check",
    category="layer2",
    series="condition",
    order=LAYER2_BINARY_ORDER,
    series_order=CONDITION_ORDER,
    ylabel="instances",
    title="Layer 2, binary properties, summed over properties, examples and replicates",
)

In [ ]:
plot_stacked_counts(
    layer_2[layer_2["layer2"].isin(LAYER2_GRADED_ORDER)],
    group="check",
    category="layer2",
    series="condition",
    order=LAYER2_GRADED_ORDER,
    series_order=CONDITION_ORDER,
    ylabel="instances",
    title="Layer 2, graded and multiclass properties, summed over properties, examples and replicates",
)

## The verdict rule

Each fan-out condition is compared with its per-check control, paired by
example and bootstrapped over the 38 examples.

- **Non-inferior** when the lower bound of the 95% CI lies above **−δ**
- **Degraded** when the interval lies entirely below −δ
- **Inconclusive** when it straddles −δ

Six comparisons — three checks × two arrangements — each running the gate
sequence independently, and all reported whatever they say.

In [ ]:
def verdict(low, high, delta):
    """Read one interval against one margin."""
    if pd.isna(low) or pd.isna(high):
        return "no data"
    if low > -delta:
        return "non-inferior"
    if high < -delta:
        return "degraded"
    return "inconclusive"


def rate_gate(frame, *, category, numerator, denominator, group, delta, label):
    """One rate endpoint, for each fan-out condition against its control."""
    out = []
    for check in CHECKS:
        sub = frame[frame["check"] == check]
        for fan_out, control in PAIRS.items():
            pair = sub[sub["condition"].isin((fan_out, control))]
            if pair["condition"].nunique() < 2:
                continue
            contrast = rate_contrast(
                pair.drop(columns=["arm"]).rename(columns={"condition": "arm"}),
                baseline=control,
                variant=fan_out,
                category=category,
                numerator=numerator,
                denominator=denominator,
                group=group,
                seed=SEED,
            )
            contrast.insert(0, "condition", fan_out)
            contrast.insert(0, "endpoint", label)
            out.append(contrast)
    result = pd.concat(out, ignore_index=True)
    result["half_width"] = (result["ci_high"] - result["ci_low"]) / 2
    result["verdict"] = [
        verdict(lo, hi, delta) for lo, hi in zip(result["ci_low"], result["ci_high"])
    ]
    return result

## Gate 1 — layer S: did each check return the right row set?

`correct_row` as a fraction of gold rows, per check. Under D each check still
answers its own list, so a panel one check drops is a row that check alone
loses — which is what would show if one check's reading of the figure shaped
another's.

In [ ]:
gate1 = rate_gate(
    layer_s,
    category="outcome",
    numerator=("correct_row",),
    denominator=("correct_row", "missing_row"),
    group=("check", "list_key"),
    delta=DELTA["layer S"],
    label="layer S",
)
gate1[[
    "condition", "check", "difference", "ci_low", "ci_high",
    "half_width", "n_examples", "paired_fraction", "verdict",
]]

## Gate 2 — layer 1: did it judge applicability the same way?

`correct_applicable` + `correct_NA` over all profiled instances. Where a
classification is shared across checks — one `classify-panels` call serving
three — this is where it would show.

In [ ]:
gate2 = rate_gate(
    layer_1,
    category="layer1",
    numerator=("correct_applicable", "correct_NA"),
    denominator=None,
    group=("check",),
    delta=DELTA["layer 1"],
    label="layer 1",
)
gate2[[
    "condition", "check", "difference", "ci_low", "ci_high",
    "half_width", "n_examples", "paired_fraction", "verdict",
]]

## Gate 3 — layer 2: the check-specific judgement

`mean_score` per property, on applicable instances, paired by example with
replicates averaged first. Per property, never pooled. The interval is
`difference ± 1.96·se`, as in exp-02.

In [ ]:
rows = []
for check in CHECKS:
    sub = scores[scores["check"] == check]
    for fan_out, control in PAIRS.items():
        pair = sub[sub["condition"].isin((fan_out, control))]
        if pair["condition"].nunique() < 2:
            continue
        contrast = arm_contrast(
            pair.drop(columns=["arm"]).rename(columns={"condition": "arm"}),
            baseline=control,
            variant=fan_out,
        )
        contrast.insert(0, "condition", fan_out)
        rows.append(contrast)

gate3 = pd.concat(rows, ignore_index=True)
gate3["half_width"] = 1.96 * gate3["se"].astype(float)
gate3["ci_low"] = gate3["difference"].astype(float) - gate3["half_width"]
gate3["ci_high"] = gate3["difference"].astype(float) + gate3["half_width"]
gate3["resolves_delta"] = gate3["half_width"] < DELTA["layer 2"]
gate3["verdict"] = [
    verdict(lo, hi, DELTA["layer 2"])
    for lo, hi in zip(gate3["ci_low"], gate3["ci_high"])
]
gate3.sort_values("difference")[[
    "condition", "check", "property", "difference", "ci_low", "ci_high",
    "half_width", "resolves_delta", "n_examples", "paired_fraction", "verdict",
]]

## The six claims, side by side

One row per condition per gate, with D's dispatch rate beside it: it qualifies
what a verdict means, it does not decide whether the verdict counts.

In [ ]:
def summarise(frame, endpoint, delta):
    g = frame.groupby(["condition", "check"])["verdict"].value_counts().unstack(fill_value=0)
    g.insert(0, "delta", delta)
    g.insert(0, "endpoint", endpoint)
    return g


dispatch = (
    traces[traces["part"] == "fan-out"]
    .groupby(["condition", "check"])["dispatched"].mean().rename("dispatch")
)
summary = pd.concat([
    summarise(gate1, "layer S", DELTA["layer S"]),
    summarise(gate2, "layer 1", DELTA["layer 1"]),
    summarise(gate3[gate3["resolves_delta"]], "layer 2", DELTA["layer 2"]),
]).fillna(0)
summary.join(dispatch)

### Secondary — does the fan-out change exp-02's own contrast?

`(A|B <- C_i <- D) − (A|B|C_i <- D)` against `(A|B <- C_i) − (A|B|C_i)`, per
check, for layers S and 1: whether delegating classification matters more or
less once three checks could share the result. A difference-in-differences of
per-example rates, bootstrapped over examples. **Reported, not a gate.**

In [ ]:
def per_example_rate(frame, category, numerator, denominator=None):
    """Rate per (condition, check, example), replicates pooled first."""
    counts = (
        frame.groupby(["condition", "check", "example", category])["count"].sum()
        .unstack(fill_value=0)
    )
    num = counts[[c for c in numerator if c in counts]].sum(axis=1)
    den_cols = [c for c in (denominator or counts.columns) if c in counts]
    den = counts[den_cols].sum(axis=1)
    return (num / den.where(den > 0)).rename("rate").reset_index()


def did(rates, check, rng, resamples=10_000):
    wide = rates[rates["check"] == check].pivot(
        index="example", columns="condition", values="rate"
    ).dropna()
    need = [f"{DELEGATING} <- D", f"{MONOLITH} <- D", DELEGATING, MONOLITH]
    if not set(need) <= set(wide.columns) or wide.empty:
        return None
    d = ((wide[need[0]] - wide[need[1]]) - (wide[need[2]] - wide[need[3]])).to_numpy()
    boots = rng.choice(d, size=(resamples, len(d)), replace=True).mean(axis=1)
    return {"check": check, "did": d.mean(),
            "ci_low": np.percentile(boots, 2.5), "ci_high": np.percentile(boots, 97.5),
            "n_examples": len(d)}


rng = np.random.default_rng(SEED)
rows = []
for label, rates in (
    ("layer S", per_example_rate(layer_s, "outcome", ("correct_row",),
                                 ("correct_row", "missing_row"))),
    ("layer 1", per_example_rate(layer_1, "layer1",
                                 ("correct_applicable", "correct_NA"))),
):
    for check in CHECKS:
        row = did(rates, check, rng)
        if row:
            rows.append({"endpoint": label, **row})
pd.DataFrame(rows)

## Cost and time

Every session records `total_cost_usd`, `num_turns`, `duration_ms` and its
token breakdown in `intermediates/tool_audit.json`. The unit is **one figure**:
a fan-out session is compared with the three per-check sessions of the same
arrangement on the same example.

Replicates are averaged per `(condition, example)` first. Replicate indices are
not paired across run types — `rep-02` of the fan-out and `rep-02` of a control
are unrelated samples — so pairing is by example only, as everywhere else.

Two time baselines: the **sum** of the three sessions is what the work costs,
the **max** is how long a figure takes if the per-check sessions run at once.

**Predicted: cost and input tokens below 1.** Not predicted for output tokens or
time. Secondary, not a gate: a cheaper check that degrades is still degraded.

In [ ]:
def read_usage(path: Path) -> dict:
    audit = json.loads(path.read_text(encoding="utf-8"))
    usage = audit.get("usage") or {}
    tokens = usage.get("usage") or {}
    return {
        "cost": usage.get("total_cost_usd"),
        "turns": usage.get("num_turns"),
        "seconds": (usage.get("duration_ms") or 0) / 1000,
        "input_tokens": sum(
            tokens.get(k) or 0
            for k in ("input_tokens", "cache_creation_input_tokens", "cache_read_input_tokens")
        ),
        "cache_read_tokens": tokens.get("cache_read_input_tokens") or 0,
        "output_tokens": tokens.get("output_tokens") or 0,
    }


rows = []
for part, checklist, check, root in run_roots():
    if not root.is_dir():
        continue
    for arm, replicate, leaf in iter_leaves(root):
        for audit in leaf.rglob("intermediates/tool_audit.json"):
            rows.append({
                "part": part,
                "arrangement": MONOLITH if arm == BASELINE_ARM else DELEGATING,
                "check": check,
                "replicate": replicate,
                "example": str(audit.parent.parent.relative_to(leaf)),
                **read_usage(audit),
            })
usage = pd.DataFrame(rows)

# Replicates averaged per (part, arrangement, check, example) first.
mean = usage.groupby(["part", "arrangement", "check", "example"]).mean(numeric_only=True)
mean = mean.drop(columns="replicate").reset_index()

QUANTITIES = ("cost", "input_tokens", "cache_read_tokens", "output_tokens", "turns", "seconds")
fan = mean[mean["part"] == "fan-out"].set_index(["arrangement", "example"])
per = mean[mean["part"] == "per-check"]
# A figure's per-check baseline needs all three checks on that example.
complete = per.groupby(["arrangement", "example"])["check"].nunique() == len(CHECKS)
per = per.set_index(["arrangement", "example"]).loc[complete[complete].index]
baseline = per.groupby(level=[0, 1])[list(QUANTITIES)].sum()
baseline["seconds_max"] = per.groupby(level=[0, 1])["seconds"].max()

figure = fan[list(QUANTITIES)].join(baseline, rsuffix="_per_check", how="inner")
print(f"{len(figure)} (arrangement, example) figures with both a fan-out and a full per-check baseline")
figure.groupby(level=0).mean()

In [ ]:
def ratio_interval(num, den, rng, resamples=10_000):
    """Mean of per-example ratios, with a percentile bootstrap over examples."""
    r = (num / den).replace([np.inf, -np.inf], np.nan).dropna().to_numpy()
    if not len(r):
        return (np.nan, np.nan, np.nan, 0)
    boots = rng.choice(r, size=(resamples, len(r)), replace=True).mean(axis=1)
    return (r.mean(), np.percentile(boots, 2.5), np.percentile(boots, 97.5), len(r))


rng = np.random.default_rng(SEED)
rows = []
for arrangement, sub in figure.groupby(level=0):
    for quantity in QUANTITIES:
        ratio, low, high, n = ratio_interval(sub[quantity], sub[f"{quantity}_per_check"], rng)
        rows.append({"arrangement": arrangement, "quantity": quantity,
                     "ratio": ratio, "ci_low": low, "ci_high": high, "n_examples": n})
    ratio, low, high, n = ratio_interval(sub["seconds"], sub["seconds_max"], rng)
    rows.append({"arrangement": arrangement, "quantity": "seconds vs max",
                 "ratio": ratio, "ci_low": low, "ci_high": high, "n_examples": n})
costs = pd.DataFrame(rows)
costs["below_1"] = costs["ci_high"] < 1
costs

## What this does not do

It does not decide whether the experiment succeeded, and it computes no reading
of dispatch against the gates — that interpretation is fixed in
[`thinking/experiments/exp-03-checklist-entry.md`](../../thinking/experiments/exp-03-checklist-entry.md)
before the run, where it can be read and disagreed with.